# Shape Sensitivities

`Sensitivity=True` gives `Mesh.dUdx`, the derivative of the displacement vector
with respect to the mesher's design variables. These are analytical derivatives
of the discretised problem, not finite differences.

In [1]:
import contextlib, io

import numpy as np

from fem2d.meshers import Mesh, Q8Mesh
from fem2d.solvers import FEMSolvers

quiet = lambda: contextlib.redirect_stdout(io.StringIO())

E, v, t, plane = 210000.0, 0.3, 1.0, 1
el_num, Load = 6, -25.0


def cantilever(ElementType, Length=10.0, Height=0.5):
    Mesher = Q8Mesh if ElementType == 'Q8' else Mesh
    m = Mesher(E, v, t, plane, ElementType)
    m.SimpleBeam(el_num, Length, Height, Load)
    return m

## Design variables

Each mesher declares how many design variables it has and how every nodal
co-ordinate moves with them. `SimpleBeam` uses Length and Height; the eight node
version differentiates Length only.

In [2]:
for ElementType in ('Q4', 'Q8'):
    m = cantilever(ElementType)
    print(f"{ElementType}: VariableNumber={m.VariableNumber}, "
          f"dXdx{m.dXdx.shape} (nodes x 2*variables)")

Q4: VariableNumber=2, dXdx(14, 4) (nodes x 2*variables)
Q8: VariableNumber=1, dXdx(33, 2) (nodes x 2*variables)


## All elements, all solvers

Checked against a central finite difference of the whole solve. The finite
difference step is relative, since a step that suits Length=10 is far too coarse
for Height=0.5.

In [3]:
Solvers = {
    'Linear':    lambda m, S: FEMSolvers.LinearSolver(m, Sensitivity=S),
    'NonLinear': lambda m, S: FEMSolvers.NonLinearSolver(m, LoadSteps=2, MaxIter=20, tol=1e-8, Sensitivity=S),
    'ArcLength': lambda m, S: FEMSolvers.ArcLengthSolver(m, ArcLength=0.5, TotalArcLength=1.5, tol=1e-8, MaxIter=20, Sensitivity=S),
}
Base = (10.0, 0.5)
Names = ('Length', 'Height')


def CentralDifference(ElementType, run, var, rel=1e-4):
    """Central difference of the tip displacement w.r.t design variable var."""
    step = rel*Base[var]
    Ends = []
    for sign in (+1, -1):
        x = list(Base)
        x[var] += sign*step
        m = cantilever(ElementType, *x)
        with quiet():
            run(m, False)
        Ends.append(m.U[-1, 0])
    return (Ends[0] - Ends[1])/(2*step)


print(f"{'element':<9}{'solver':<12}{'variable':<9}{'analytic':>14}{'central FD':>14}{'rel err':>10}")
for ElementType in ('Q4', '5B', 'Q8'):
    for name, run in Solvers.items():
        m = cantilever(ElementType)
        with quiet():
            run(m, True)
        for var in range(m.VariableNumber):
            Analytic = m.dUdx[-1, var]
            FD = CentralDifference(ElementType, run, var)
            print(f"{ElementType:<9}{name:<12}{Names[var]:<9}"
                  f"{Analytic:>14.5e}{FD:>14.5e}{abs(Analytic - FD)/abs(FD):>10.1e}")

element  solver      variable       analytic    central FD   rel err
Q4       Linear      Length     -1.00184e-01  -1.00184e-01   2.5e-10
Q4       Linear      Height      2.00368e+00   2.00368e+00   2.0e-08


Q4       NonLinear   Length     -9.97658e-02  -9.97658e-02   2.0e-10


Q4       NonLinear   Height      1.99011e+00   1.99011e+00   8.6e-09


Q4       ArcLength   Length     -8.86468e-05  -8.86468e-05   1.1e-07


Q4       ArcLength   Height      1.10343e-03   1.10343e-03   4.3e-09
5B       Linear      Length     -1.03339e+00  -1.03339e+00   1.8e-10
5B       Linear      Height      2.06678e+01   2.06678e+01   1.9e-08


...

5B       ArcLength   Length     -8.37009e-05  -8.37009e-05   6.6e-08


5B       ArcLength   Height      1.04381e-03   1.04381e-03   5.1e-09


Q8       Linear      Length     -1.03565e+00  -1.03565e+00   4.4e-07


Q8       NonLinear   Length     -8.27467e-01  -8.27467e-01   3.3e-09


Q8       ArcLength   Length     -5.64181e-05  -5.64180e-05   5.1e-07


## Along an arc length path

The arc length solver also returns the sensitivity of the **load factor**,
`dLdx`, and keeps the per step history in `dUdx_All` and `dLdx`. `Mesh.dUdx` is
the final step.

In [4]:
m = cantilever('5B')
with quiet():
    Solvers['ArcLength'](m, True)

print(f"arc steps      : {len(m.LoadValues) - 1}")
print(f"dUdx_All shape : {m.dUdx_All.shape}   (dof x variables x step)")
print(f"final dUdx     : {m.dUdx[-1]}")
print(f"\ndLdx per step (rows), columns are {Names}:\n{m.dLdx}")

arc steps      : 3
dUdx_All shape : (28, 2, 3)   (dof x variables x step)
final dUdx     : [-8.37009350e-05  1.04381059e-03]

dLdx per step (rows), columns are ('Length', 'Height'):
[[-0.02189586  0.43795154]
 [-0.04380626  0.87568621]
 [-0.06584242  1.31452771]]


In [5]:
# The load factor sensitivity against a finite difference of the final load factor.
def LoadFactorFD(var, rel=1e-4):
    step = rel*Base[var]
    Ends = []
    for sign in (+1, -1):
        x = list(Base)
        x[var] += sign*step
        mm = cantilever('5B', *x)
        with quiet():
            Solvers['ArcLength'](mm, False)
        Ends.append(mm.LoadValues[-1])
    return (Ends[0] - Ends[1])/(2*step)


print(f"{'variable':<9}{'analytic dLdx':>16}{'central FD':>14}{'rel err':>10}")
for var in range(m.VariableNumber):
    Analytic, FD = m.dLdx[-1, var], LoadFactorFD(var)
    print(f"{Names[var]:<9}{Analytic:>16.5e}{FD:>14.5e}{abs(Analytic - FD)/abs(FD):>10.1e}")

variable    analytic dLdx    central FD   rel err


Length       -6.58424e-02  -6.58424e-02   3.4e-08


Height        1.31453e+00   1.31453e+00   3.1e-09


## Other meshers

`LeeFrame` takes the two member lengths as design variables, `SemiCircularArch`
takes the design radii, so its variable count follows the length of `r_design`.

In [6]:
Cases = {
    'LeeFrame':         (lambda m: m.LeeFrame(10, [10.0, 10.0], Load),                  ('length up', 'length side')),
    'SemiCircularArch': (lambda m: m.SemiCircularArch(12, 10.0, [10.0, 10.0], 1.0, Load), ('r1', 'r2')),
}

for name, (build, labels) in Cases.items():
    m = Mesh(E, v, t, plane, '5B')
    build(m)
    with quiet():
        FEMSolvers.LinearSolver(m, Sensitivity=True)
    Probe = int(np.argmax(np.abs(m.U)))
    print(f"{name}: {m.VariableNumber} variables, most displaced dof {Probe}")
    for var, label in enumerate(labels):
        print(f"    dU/d({label}) = {m.dUdx[Probe, var]: .6e}")

LeeFrame: 2 variables, most displaced dof 29
    dU/d(length up) = -8.829869e-05
    dU/d(length side) = -3.776800e-04


SemiCircularArch: 2 variables, most displaced dof 38
    dU/d(r1) =  2.017013e-03
    dU/d(r2) = -3.130767e-02
